# Test chỉ số thời gian time_decay

In [1]:
import numpy as np
import pandas as pd
import time

In [2]:
from surprise import SVD, CoClustering, NMF, SlopeOne, KNNBaseline, BaselineOnly, Dataset, Reader
from surprise.model_selection import train_test_split
from surprise import accuracy
from surprise.model_selection import cross_validate

In [3]:
path = "movie_data/"

In [4]:
df_items = pd.read_csv(path + "movies.csv")
df_interactions = pd.read_csv(path + "ratings.csv")

In [5]:
df_items.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [6]:
df_interactions.head()

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [7]:
from collections import Counter

# Algorithm 1: Phân loại thể loại yêu thích và không thích của người dùng

def get_user_genres_preferences(user_id, df_interactions, df_items):
  # Get movieId which user rated
  user_ratings = df_interactions[df_interactions['userId'] == user_id]

  # Count genre
  genre_counts = Counter()
  for m_id in user_ratings['movieId']:
    # Get string genres of movie ("Action|Adventure")
    genre_str = df_items.loc[df_items['movieId'] == m_id, 'genres'].values[0]
    if pd.isna(genre_str): continue
    for genre in genre_str.split('|'):
      genre_counts[genre] += 1

  # Get all genre
  all_genres = set()
  df_items['genres'].str.split('|').apply(lambda x: all_genres.update(x) if isinstance(x, list) else None)

  # Sort asc
  sorted_genres = sorted(list(all_genres), key=lambda x: genre_counts.get(x, 0), reverse=True)

  # Find mid-point
  mid_point = len(sorted_genres) // 2
  fav_genres = sorted_genres[:mid_point]
  unfav_genres = sorted_genres[mid_point:]

  return fav_genres, unfav_genres


In [8]:
# Algorithm 2: Kiểm tra tính liên quan (Relevance) dựa trên Genre
def is_movie_relevant(movie_id, df_items, fav_genres, unfav_genres):
    res = df_items.loc[df_items['movieId'] == movie_id, 'genres'].values
    if len(res) == 0 or pd.isna(res[0]): return False

    movie_genres = [g.strip() for g in res[0].split('|')]
    # Score = (Số genre thích) - (Số genre ghét)
    score = sum(1 for g in movie_genres if g in fav_genres) - \
            sum(1 for g in movie_genres if g in unfav_genres)
    return score > 0

In [9]:
# Algorithm 4: Tính các chỉ số Precision@K, Recall@K, F1 score
def calculate_metrics_at_k(predictions, df_interaction, df_items, k):
  user_est_true = {}
  for uid, iid, true_r, est, _ in predictions:
    if uid not in user_est_true:
      user_est_true[uid] = []
    user_est_true[uid].append((iid, est))

  precisions = []
  recalls = []

  for uid, user_ratings in user_est_true.items():
    # Lấy gu của user
    fav, unfav = get_user_genres_preferences(uid, df_interaction, df_items)

    # Sắp xếp dự đoán theo điểm 'est' cao nhất
    user_ratings.sort(key=lambda x: x[1], reverse=True)
    top_k = user_ratings[:k]

    # Tính TP (True Positives)
    n_rel_and_rec_k = sum(1 for (iid, est) in top_k if is_movie_relevant(iid, df_items, fav, unfav))

    # Tính tổng số phim relevant thực tế có trong tập test của user
    n_rel = sum(1 for (iid, est) in user_ratings if is_movie_relevant(iid, df_items, fav, unfav))

    # Tính precision and recall cho user
    precisions.append(n_rel_and_rec_k / k)
    recalls.append(n_rel_and_rec_k / n_rel if n_rel != 0 else 0)

  # Lấy avg
  avg_prec = np.mean(precisions)
  avg_rec = np.mean(recalls)

  f1 = (2 * avg_prec * avg_rec) / (avg_rec + avg_prec) if (avg_rec + avg_prec) >0 else 0

  return avg_prec, avg_rec, f1

In [ ]:
# # Danh sách các thuật toán
# algorithms = [
#     SVD(),
#     CoClustering(),
#     NMF(),
#     SlopeOne(),
#     KNNBaseline(),
#     BaselineOnly()
# ]

In [ ]:
# Danh sách các thuật toán
algorithms = [
    SVD(n_factors=50, n_epochs=20, lr_all=0.005, reg_all=0.02),
    
]

In [12]:
# Chia dữ liệu Trainning và test, tỉ lệ 75/25
reader = Reader(rating_scale=(0.5, 5))

data = Dataset.load_from_df(df_interactions[['userId', 'movieId', 'rating']], reader)

trainset, testset = train_test_split(data, test_size=0.25)

In [13]:
from surprise import accuracy
from collections import Counter

In [14]:
# Huấn luyện, dự đoán và đo thời gian
results_table = []

for algorithm in algorithms:
  algo_name = algorithm.__class__.__name__

  start_time = time.time()

  # Train
  algorithm.fit(trainset)

  # Predict
  predictions = algorithm.test(testset)

  #Time
  eval_time = time.time() - start_time


  # Tinh RMSE va MAE
  rmse_val = accuracy.rmse(predictions, verbose=False)
  mae_val = accuracy.mae(predictions, verbose=False)

  # Tinh Precision, recall, f1
  prec, rec, f1 = calculate_metrics_at_k(predictions, df_interactions, df_items, k = 15)

  results_table.append([algo_name, rmse_val, mae_val, prec, rec, f1, eval_time])
  print(f"{algo_name:<15} | {rmse_val:<8.4f} | {mae_val:<8.4f} | {prec:<10.4f} | {rec:<8.4f} | {f1:<8.4f} | {eval_time:<8.4f}")


TypeError: 'SVD' object is not iterable

In [ ]:

# Xuất ra DataFrame để hiển thị đẹp
df_final_results = pd.DataFrame(results_table, columns=['Algorithms', 'RMSE', 'MAE', 'Precision', 'Recall', 'F1 Score', 'Time'])
df_final_results

,Algorithms,RMSE,MAE,Precision,Recall,F1 Score,Time
0,SVD,0.882464,0.677149,0.736721,0.681050,0.707793,0.872243
1,CoClustering,0.955874,0.739092,0.731694,0.679848,0.704819,2.033706
2,NMF,0.925620,0.708709,0.734645,0.681016,0.706814,1.549831
3,SlopeOne,0.909703,0.694513,0.731475,0.679723,0.704650,6.668523
4,KNNBaseline,0.884661,0.675019,0.728634,0.679649,0.703289,1.678330
5,BaselineOnly,0.881295,0.678120,0.734754,0.680622,0.706653,0.263182
